In [0]:
%run /Users/mathisneha2004@gmail.com/config/Pipeline_Config

In [0]:
from datetime import datetime
gold_start = datetime.now()
gold_input = spark.table(SILVER_CLAIMS_TABLE).count()
print(f"{'='*50}")
print(f"GOLD LAYER STARTED")
print(f"{'='*50}")
print(f"Start Time    : {gold_start}")
print(f"Input Records : {gold_input:,}")
print(f"{'='*50}")

# Gold Layer - Dimensional Model

Create dimensional model (star schema) from Silver Layer tables:
* **Source Tables**: healthcare_claims_data.silver.claims, healthcare_claims_data.silver.hospital
* **Dimension Tables**: dim_patient, dim_disease, dim_claim, dim_geography, dim_date
* **Fact Table**: fact_claims

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

In [0]:
# State code mapping: Abbreviation -> Numeric Code
state_mapping = {
    'AL': '01', 'AK': '02', 'AZ': '04', 'AR': '05', 'CA': '06', 'CO': '08', 'CT': '09',
    'DE': '10', 'DC': '11', 'FL': '12', 'GA': '13', 'HI': '15', 'ID': '16', 'IL': '17',
    'IN': '18', 'IA': '19', 'KS': '20', 'KY': '21', 'LA': '22', 'ME': '23', 'MD': '24',
    'MA': '25', 'MI': '26', 'MN': '27', 'MS': '28', 'MO': '29', 'MT': '30', 'NE': '31',
    'NV': '32', 'NH': '33', 'NJ': '34', 'NM': '35', 'NY': '36', 'NC': '37', 'ND': '38',
    'OH': '39', 'OK': '40', 'OR': '41', 'PA': '42', 'RI': '44', 'SC': '45', 'SD': '46',
    'TN': '47', 'TX': '48', 'UT': '49', 'VT': '50', 'VA': '51', 'WA': '53', 'WV': '54',
    'WI': '55', 'WY': '56'
}

# Create mapping expression for Spark
from pyspark.sql.functions import create_map, lit
from itertools import chain

state_mapping_expr = create_map([lit(x) for x in chain(*state_mapping.items())])

print(f"✓ State code mapping created: {len(state_mapping)} states")

## STEP 1 - Read Silver Tables
Read and filter both silver tables where DATA_QUALITY_FLAG = 'PASS'

In [0]:
try:
    print("STEP 1: Reading Silver tables...\n")

    claims_silver_df = spark.table(SILVER_CLAIMS_TABLE)
    hospital_silver_df = spark.table(SILVER_HOSPITAL_TABLE)

    claims_clean_df = claims_silver_df.filter(F.col("DATA_QUALITY_FLAG") == DATA_QUALITY_PASS)
    hospital_clean_df = hospital_silver_df.filter(F.col("DATA_QUALITY_FLAG") == DATA_QUALITY_PASS)

    claims_count = claims_clean_df.count()
    hospital_count = hospital_clean_df.count()

    print(f"✓ STEP 1 Complete")
    print(f"  Claims records (PASS): {claims_count:,}")
    print(f"  Hospital records (PASS): {hospital_count:,}\n")
except Exception as e:
    print(f"✗ STEP 1 Failed: {str(e)}")
    raise

## STEP 2 - Aggregate Hospital by State
Aggregate hospital data to one row per state

In [0]:
try:
    print("STEP 2: Aggregating hospital data by state...\n")

    # Convert state abbreviations to numeric codes for joining with claims
    hospital_with_code = hospital_clean_df.withColumn(
        "STATE_CODE_NUMERIC",
        state_mapping_expr[F.col("Rndrng_Prvdr_State_Abrvtn")]
    )

    hospital_agg_df = hospital_with_code.groupBy("STATE_CODE_NUMERIC").agg(
        F.round(F.avg("Avg_Mdcr_Pymt_Amt"), 2).alias("STATE_AVG_MEDICARE_PAYMENT"),
        F.sum("Tot_Dschrgs").alias("STATE_TOTAL_DISCHARGES"),
        F.countDistinct("Rndrng_Prvdr_CCN").alias("STATE_TOTAL_HOSPITALS"),
        F.first("LOCATION_TYPE").alias("STATE_LOCATION_TYPE")
    ).withColumnRenamed("STATE_CODE_NUMERIC", "STATE_ABBR")

    hospital_agg_count = hospital_agg_df.count()

    print(f"✓ STEP 2 Complete")
    print(f"  Hospital aggregated records: {hospital_agg_count:,}")
    print(f"  State codes converted to numeric format\n")
except Exception as e:
    print(f"✗ STEP 2 Failed: {str(e)}")
    raise

## STEP 3 - Calculate National Average
Calculate national average Medicare payment

In [0]:
try:
    print("STEP 3: Calculating national average...\n")

    NATIONAL_AVG = hospital_agg_df.select(F.avg("STATE_AVG_MEDICARE_PAYMENT")).first()[0]

    print(f"✓ STEP 3 Complete")
    print(f"  National average Medicare payment: ${NATIONAL_AVG:,.2f}\n")
except Exception as e:
    print(f"✗ STEP 3 Failed: {str(e)}")
    raise

## STEP 4 - Add Benchmark Category
Add GOLD_BENCHMARK_CATEGORY to hospital aggregated data

In [0]:
try:
    print("STEP 4: Adding benchmark category...\n")

    hospital_agg_df = hospital_agg_df.withColumn(
        "GOLD_BENCHMARK_CATEGORY",
        F.when(F.col("STATE_AVG_MEDICARE_PAYMENT") > NATIONAL_AVG * GOLD_BENCHMARK_HIGH_THRESHOLD, "Above National Average")
         .when(F.col("STATE_AVG_MEDICARE_PAYMENT") < NATIONAL_AVG * GOLD_BENCHMARK_LOW_THRESHOLD, "Below National Average")
         .otherwise("At National Average")
    )

    print(f"✓ STEP 4 Complete")
    print(f"  GOLD_BENCHMARK_CATEGORY added\n")
except Exception as e:
    print(f"✗ STEP 4 Failed: {str(e)}")
    raise

## STEP 5 - Join Claims with Hospital
Join claims with aggregated hospital data

In [0]:
try:
    print("STEP 5: Joining claims with hospital data...\n")

    # Drop STATE_AVG_MEDICARE_PAYMENT from claims_clean_df to avoid ambiguous reference
    # We want to use the hospital-aggregated version from hospital_agg_df
    claims_for_join = claims_clean_df.drop("STATE_AVG_MEDICARE_PAYMENT")
    
    joined_df = claims_for_join.join(
        hospital_agg_df,
        claims_for_join.SP_STATE_CODE == hospital_agg_df.STATE_ABBR,
        "left"
    ).drop("STATE_ABBR").filter(
        F.col("STATE_AVG_MEDICARE_PAYMENT").isNotNull()
    )

    joined_count = joined_df.count()

    print(f"✓ STEP 5 Complete")
    print(f"  Joined records: {joined_count:,}\n")
except Exception as e:
    print(f"✗ STEP 5 Failed: {str(e)}")
    raise

## STEP 6 - Create dim_patient (SCD Type 2)
One row per unique patient with history tracking

**SCD Type 2 Implementation:**
* **PATIENT_SK**: Surrogate key (auto-incrementing)
* **PATIENT_ID**: Business key (natural key)
* **EFFECTIVE_START_DATE**: When this version became active
* **EFFECTIVE_END_DATE**: When this version expired (NULL for current)
* **IS_CURRENT**: Boolean flag (True = current version)

**Logic:**
* First load: All records inserted as current with IS_CURRENT = True
* Subsequent loads:
  * Unchanged records: Keep as-is
  * Changed records: Expire old version (set EFFECTIVE_END_DATE, IS_CURRENT = False), insert new version with new surrogate key
  * New records: Insert as current with IS_CURRENT = True

In [0]:
try:
    print("STEP 6: Creating dim_patient with SCD Type 2...\n")
    from pyspark.sql import Window
    from datetime import date

    # Prepare new patient data
    new_patients = joined_df.select(
        "PATIENT_ID", "BENE_SEX_IDENT_CD", "BENE_RACE_CD", "BENE_ESRD_IND",
        "SP_STATE_CODE", "BENE_COUNTY_CD", "BENE_BIRTH_DT", "BENE_DEATH_DT",
        "AGE", "AGE_GROUP", "BENE_HI_CVRAGE_TOT_MONS", "BENE_SMI_CVRAGE_TOT_MONS",
        "BENE_HMO_CVRAGE_TOT_MONS", "PLAN_CVRG_MOS_NUM"
    ).dropDuplicates(["PATIENT_ID"])

    target_table = GOLD_DIM_PATIENT
    today = date.today()

    # Check if table exists
    table_exists = spark.catalog.tableExists(target_table)
    
    # If table exists, check if it has SCD Type 2 structure
    has_scd_structure = False
    if table_exists:
        existing_columns = [field.name for field in spark.table(target_table).schema.fields]
        has_scd_structure = "PATIENT_SK" in existing_columns
    
    if not table_exists or not has_scd_structure:
        # First load or migration - create/recreate table with SCD Type 2 columns
        if table_exists and not has_scd_structure:
            print("  Existing table found without SCD Type 2 structure - migrating...")
        else:
            print("  First load - creating table with SCD Type 2 structure...")
        
        dim_patient = new_patients \
            .withColumn("PATIENT_SK", F.monotonically_increasing_id()) \
            .withColumn("EFFECTIVE_START_DATE", F.lit(today).cast("date")) \
            .withColumn("EFFECTIVE_END_DATE", F.lit(None).cast("date")) \
            .withColumn("IS_CURRENT", F.lit(True))

        dim_patient.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)
        
        patient_count = dim_patient.count()
        print(f"  ✓ Initial load complete: {patient_count:,} records\n")
    else:
        # SCD Type 2 merge logic
        print("  Performing SCD Type 2 merge...")
        
        # Read existing dimension
        existing_dim = spark.table(target_table).filter(F.col("IS_CURRENT") == True)
        
        # Get max surrogate key for new records
        max_sk = spark.table(target_table).agg(F.max("PATIENT_SK")).first()[0] or 0
        
        # Define SCD Type 2 attributes (all except PATIENT_ID)
        scd_columns = [
            "BENE_SEX_IDENT_CD", "BENE_RACE_CD", "BENE_ESRD_IND",
            "SP_STATE_CODE", "BENE_COUNTY_CD", "BENE_BIRTH_DT", "BENE_DEATH_DT",
            "AGE", "AGE_GROUP", "BENE_HI_CVRAGE_TOT_MONS", "BENE_SMI_CVRAGE_TOT_MONS",
            "BENE_HMO_CVRAGE_TOT_MONS", "PLAN_CVRG_MOS_NUM"
        ]
        
        # Join new with existing to find changes
        comparison = new_patients.alias("new").join(
            existing_dim.alias("old"),
            F.col("new.PATIENT_ID") == F.col("old.PATIENT_ID"),
            "left"
        )
        
        # Identify changed records (at least one SCD attribute differs)
        change_condition = None
        for col in scd_columns:
            col_diff = (F.col(f"new.{col}") != F.col(f"old.{col}")) | \
                       (F.col(f"new.{col}").isNull() & F.col(f"old.{col}").isNotNull()) | \
                       (F.col(f"new.{col}").isNotNull() & F.col(f"old.{col}").isNull())
            change_condition = col_diff if change_condition is None else (change_condition | col_diff)
        
        # Get changed and new records
        changed_patients = comparison.filter(
            F.col("old.PATIENT_ID").isNotNull() & change_condition
        ).select("new.*")
        
        new_only_patients = comparison.filter(
            F.col("old.PATIENT_ID").isNull()
        ).select("new.*")
        
        # Expired patient IDs (changed records)
        expired_ids = changed_patients.select("PATIENT_ID").distinct()
        
        # Close expired records
        from delta.tables import DeltaTable
        delta_table = DeltaTable.forName(spark, target_table)
        
        if expired_ids.count() > 0:
            delta_table.alias("target").merge(
                expired_ids.alias("expired"),
                "target.PATIENT_ID = expired.PATIENT_ID AND target.IS_CURRENT = true"
            ).whenMatchedUpdate(set={
                "EFFECTIVE_END_DATE": F.lit(today),
                "IS_CURRENT": F.lit(False)
            }).execute()
            print(f"  Expired {expired_ids.count()} existing records")
        
        # Prepare new versions of changed records + completely new records
        records_to_insert = changed_patients.union(new_only_patients)
        
        if records_to_insert.count() > 0:
            # Assign new surrogate keys
            window = Window.orderBy(F.monotonically_increasing_id())
            records_to_insert = records_to_insert \
                .withColumn("_row_num", F.row_number().over(window)) \
                .withColumn("PATIENT_SK", F.col("_row_num") + max_sk) \
                .drop("_row_num") \
                .withColumn("EFFECTIVE_START_DATE", F.lit(today).cast("date")) \
                .withColumn("EFFECTIVE_END_DATE", F.lit(None).cast("date")) \
                .withColumn("IS_CURRENT", F.lit(True))
            
            # Append new records
            records_to_insert.write.format("delta") \
                .mode("append") \
                .saveAsTable(target_table)
            
            print(f"  Inserted {records_to_insert.count()} new/changed records")
        
        total_count = spark.table(target_table).count()
        current_count = spark.table(target_table).filter(F.col("IS_CURRENT") == True).count()
        print(f"  ✓ SCD Type 2 merge complete\n")
        print(f"  Total records (including history): {total_count:,}")
        print(f"  Current records: {current_count:,}\n")

    print(f"✓ STEP 6 Complete: {target_table}")
except Exception as e:
    print(f"✗ STEP 6 Failed: {str(e)}")
    raise

In [0]:
# Verify SCD Type 2 structure
print("="*80)
print("SCD TYPE 2 VERIFICATION - dim_patient")
print("="*80)

try:
    dim_patient_df = spark.table(GOLD_DIM_PATIENT)
    
    # Check schema includes SCD Type 2 columns
    scd_columns = ["PATIENT_SK", "EFFECTIVE_START_DATE", "EFFECTIVE_END_DATE", "IS_CURRENT"]
    schema_cols = [field.name for field in dim_patient_df.schema.fields]
    
    print("\n1. Schema Check:")
    for col in scd_columns:
        status = "✓" if col in schema_cols else "✗"
        print(f"   {status} {col}")
    
    # Summary statistics
    total_records = dim_patient_df.count()
    current_records = dim_patient_df.filter(F.col("IS_CURRENT") == True).count()
    historical_records = dim_patient_df.filter(F.col("IS_CURRENT") == False).count()
    
    print(f"\n2. Record Counts:")
    print(f"   Total records:      {total_records:,}")
    print(f"   Current records:    {current_records:,}")
    print(f"   Historical records: {historical_records:,}")
    
    # Sample data
    print(f"\n3. Sample Current Records (first 3):")
    display(dim_patient_df.filter(F.col("IS_CURRENT") == True).select(
        "PATIENT_SK", "PATIENT_ID", "AGE", "AGE_GROUP", 
        "EFFECTIVE_START_DATE", "EFFECTIVE_END_DATE", "IS_CURRENT"
    ).limit(3))
    
    print("\n" + "="*80)
    print("✓ SCD Type 2 verification complete")
    print("="*80)
except Exception as e:
    print(f"\n✗ Verification failed: {str(e)}")

## STEP 7 - Create dim_disease
One row per unique patient with disease information

In [0]:
try:
    print("STEP 7: Creating dim_disease...\n")

    dim_disease = joined_df.select(
        "PATIENT_ID", "SP_ALZHDMTA", "SP_CHF", "SP_CHRNKIDN", "SP_CNCR", "SP_COPD",
        "SP_DEPRESSN", "SP_DIABETES", "SP_ISCHMCHT", "SP_OSTEOPRS", "SP_RA_OA",
        "SP_STRKETIA", "DISEASE_COUNT", "RISK_CATEGORY"
    ).dropDuplicates(["PATIENT_ID"])

    target_table = GOLD_DIM_DISEASE

    dim_disease.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)

    disease_count = dim_disease.count()

    print(f"✓ STEP 7 Complete: {target_table}")
    print(f"  Records: {disease_count:,}\n")
except Exception as e:
    print(f"✗ STEP 7 Failed: {str(e)}")
    raise

## STEP 8 - Create dim_claim
One row per unique claim

In [0]:
try:
    print("STEP 8: Creating dim_claim...\n")

    dim_claim = joined_df.select(
        "CLM_ID", "PATIENT_ID", "CLM_FROM_DT", "CLM_THRU_DT",
        "CLAIM_DURATION_DAYS", "CLAIM_DURATION_TYPE",
        "CLAIM_STATUS", "CLAIM_TYPE",
        "ICD9_DGNS_CD_1_CLEAN", "LINE_ICD9_DGNS_CD_1_CLEAN",
        "HCPCS_CD_1", "PRF_PHYSN_NPI_1_CLEAN", "LINE_PRCSG_IND_CD_1"
    ).dropDuplicates(["CLM_ID"])

    target_table = GOLD_DIM_CLAIM

    dim_claim.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)

    claim_count = dim_claim.count()

    print(f"✓ STEP 8 Complete: {target_table}")
    print(f"  Records: {claim_count:,}\n")
except Exception as e:
    print(f"✗ STEP 8 Failed: {str(e)}")
    raise

## STEP 9 - Create dim_geography
One row per unique state

In [0]:
try:
    print("STEP 9: Creating dim_geography...\n")

    dim_geography = joined_df.select(
        "SP_STATE_CODE", "STATE_AVG_MEDICARE_PAYMENT",
        "STATE_TOTAL_DISCHARGES", "STATE_TOTAL_HOSPITALS",
        "STATE_LOCATION_TYPE", "GOLD_BENCHMARK_CATEGORY"
    ).dropDuplicates(["SP_STATE_CODE"])

    target_table = GOLD_DIM_GEOGRAPHY

    dim_geography.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)

    geography_count = dim_geography.count()

    print(f"✓ STEP 9 Complete: {target_table}")
    print(f"  Records: {geography_count:,}\n")
except Exception as e:
    print(f"✗ STEP 9 Failed: {str(e)}")
    raise

## STEP 10 - Create dim_date
One row per unique date derived from CLM_FROM_DT

In [0]:
try:
    print("STEP 10: Creating dim_date...\n")

    def parse_date_int(col_name):
        date_str = F.col(col_name).cast("string")
        return F.when(
            (F.col(col_name).isNull()) | (F.col(col_name) == 0) | (F.length(date_str) != 8),
            F.lit(None).cast("date")
        ).otherwise(
            F.to_date(date_str, "yyyyMMdd")
        )

    source_df = spark.table(SILVER_CLAIMS_TABLE).filter(F.col("DATA_QUALITY_FLAG") == DATA_QUALITY_PASS)
    
    dim_date = source_df.select("CLM_FROM_DT").dropDuplicates(["CLM_FROM_DT"])

    dim_date = dim_date.withColumn(
        "DATE_KEY",
        F.col("CLM_FROM_DT").cast(IntegerType())
    ).withColumn(
        "FULL_DATE",
        parse_date_int("CLM_FROM_DT")
    ).withColumn(
        "CLAIM_YEAR",
        F.year(F.col("FULL_DATE"))
    ).withColumn(
        "CLAIM_MONTH",
        F.month(F.col("FULL_DATE"))
    ).withColumn(
        "CLAIM_MONTH_NAME",
        F.date_format(F.col("FULL_DATE"), "MMMM")
    ).withColumn(
        "CLAIM_QUARTER",
        F.quarter(F.col("FULL_DATE"))
    ).withColumn(
        "CLAIM_QUARTER_NAME",
        F.concat(F.lit("Q"), F.quarter(F.col("FULL_DATE")))
    ).withColumn(
        "DAY_OF_WEEK",
        F.date_format(F.col("FULL_DATE"), "EEEE")
    ).withColumn(
        "IS_WEEKEND",
        F.when(F.date_format(F.col("FULL_DATE"), "E").isin(["Sat", "Sun"]), "Yes").otherwise("No")
    ).drop("CLM_FROM_DT").dropDuplicates(["DATE_KEY"])

    target_table = GOLD_DIM_DATE

    dim_date.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)

    date_count = dim_date.count()

    print(f"✓ STEP 10 Complete: {target_table}")
    print(f"  Records: {date_count:,}\n")
except Exception as e:
    print(f"✗ STEP 10 Failed: {str(e)}")
    raise

In [0]:
display(spark.table(GOLD_DIM_DATE))

## STEP 11 - Create fact_claims
One row per claim with all metrics

In [0]:
try:
    print("STEP 11: Creating fact_claims...\n")
    
    from pyspark.sql.window import Window

    # Step 1: Select all columns
    fact_raw = joined_df.select(
        "CLM_ID",
        "PATIENT_ID",
        "SP_STATE_CODE",
        "CLM_FROM_DT",
        "TOTAL_PAYMENT_AMOUNT",
        "LINE_NCH_PMT_AMT_1_CLEAN",
        "LINE_BENE_PTB_DDCTBL_AMT_1_CLEAN",
        "LINE_COINSRNC_AMT_1_CLEAN",
        "MEDREIMB_IP",
        "BENRES_IP",
        "PPPYMT_IP",
        "MEDREIMB_OP",
        "BENRES_OP",
        "PPPYMT_OP",
        "MEDREIMB_CAR",
        "BENRES_CAR",
        "PPPYMT_CAR",
        "AGE",
        "AGE_GROUP",
        "DISEASE_COUNT",
        "RISK_CATEGORY",
        "CLAIM_DURATION_DAYS",
        "CLAIM_DURATION_TYPE",
        "CLAIM_STATUS",
        "CLAIM_TYPE",
        "STATE_AVG_MEDICARE_PAYMENT",
        "STATE_TOTAL_DISCHARGES",
        "STATE_TOTAL_HOSPITALS",
        "STATE_LOCATION_TYPE",
        "GOLD_BENCHMARK_CATEGORY",
        "DATA_QUALITY_FLAG"
    )

    # Step 2: Filter out null PATIENT_ID and CLM_ID
    fact_raw = fact_raw.filter(
        F.col("PATIENT_ID").isNotNull() &
        F.col("CLM_ID").isNotNull()
    )

    # Step 3: Pick most recent claim per patient
    # Latest CLM_FROM_DT first, then latest CLM_ID as tiebreaker
    window_spec = Window.partitionBy("PATIENT_ID").orderBy(
        F.col("CLM_FROM_DT").desc(),
        F.col("CLM_ID").desc()
    )

    fact_claims = fact_raw \
        .withColumn("_rank", F.row_number().over(window_spec)) \
        .filter(F.col("_rank") == 1) \
        .drop("_rank")

    target_table = GOLD_FACT_CLAIMS

    fact_claims.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)

    fact_count = fact_claims.count()

    print(f"✓ STEP 11 Complete: {target_table}")
    print(f"  Records: {fact_count:,} (1 row per PATIENT_ID)")
    print(f"  Columns: 31 (4 FKs + 26 Measures/Attributes + 1 Flag)\n")
except Exception as e:
    print(f"✗ STEP 11 Failed: {str(e)}")
    raise

## Display All Dimension Tables
Sample data from each dimension table

In [0]:
print("=" * 80)
print("dim_patient - Patient Demographics (SCD Type 2 - Current Records Only)")
print("=" * 80)
display(spark.table(GOLD_DIM_PATIENT).filter(F.col("IS_CURRENT") == True).limit(5))

In [0]:
print("=" * 80)
print("dim_disease - Patient Disease Indicators")
print("=" * 80)
display(spark.table(GOLD_DIM_DISEASE).limit(5))

In [0]:
print("=" * 80)
print("dim_claim - Claim Details")
print("=" * 80)
display(spark.table(GOLD_DIM_CLAIM).limit(5))

In [0]:
print("=" * 80)
print("dim_geography - State-Level Metrics")
print("=" * 80)
display(spark.table(GOLD_DIM_GEOGRAPHY).limit(10))

In [0]:
print("=" * 80)
print("dim_date - Date Dimension")
print("=" * 80)
display(spark.table(GOLD_DIM_DATE).orderBy("DATE_KEY").limit(10))

In [0]:
display(spark.table(GOLD_FACT_CLAIMS).limit(10))

## Gold Layer Summary
Final verification of all gold tables

## STEP 12 - Create Materialized Views
Create aggregated materialized views for common analytical queries

In [0]:
try:
    print("Creating gold schema if not exists...\n")
    
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}")
    
    print("✓ Gold schema ready\n")
except Exception as e:
    print(f"✗ Schema creation failed: {str(e)}")
    raise

In [0]:
try:
    print("Creating mv_state_claim_summary...\n")
    
    mv_state_claim_summary = spark.sql(f"""
    SELECT 
        SP_STATE_CODE,
        COUNT(*) AS TOTAL_CLAIMS,
        SUM(TOTAL_PAYMENT_AMOUNT) AS TOTAL_PAYMENT,
        ROUND(AVG(TOTAL_PAYMENT_AMOUNT), 2) AS AVG_PAYMENT
    FROM {GOLD_FACT_CLAIMS}
    GROUP BY SP_STATE_CODE
    """)
    
    target_table = GOLD_MV_STATE
    
    mv_state_claim_summary.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)
    
    mv_count = mv_state_claim_summary.count()
    
    print(f"✓ {target_table}")
    print(f"  Records: {mv_count:,}\n")
except Exception as e:
    print(f"✗ mv_state_claim_summary failed: {str(e)}")
    raise

In [0]:
try:
    print("Creating mv_risk_category_summary...\n")
    
    mv_risk_category_summary = spark.sql(f"""
    SELECT 
        d.RISK_CATEGORY,
        COUNT(DISTINCT f.PATIENT_ID) AS TOTAL_PATIENTS,
        COUNT(*) AS TOTAL_CLAIMS,
        SUM(f.TOTAL_PAYMENT_AMOUNT) AS TOTAL_PAYMENT
    FROM {GOLD_FACT_CLAIMS} f
    JOIN {GOLD_DIM_DISEASE} d ON f.PATIENT_ID = d.PATIENT_ID
    GROUP BY d.RISK_CATEGORY
    """)
    
    target_table = GOLD_MV_RISK
    
    mv_risk_category_summary.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)
    
    mv_count = mv_risk_category_summary.count()
    
    print(f"✓ {target_table}")
    print(f"  Records: {mv_count:,}\n")
except Exception as e:
    print(f"✗ mv_risk_category_summary failed: {str(e)}")
    raise

In [0]:
try:
    print("Creating mv_claim_status_summary...\n")
    
    mv_claim_status_summary = spark.sql(f"""
    SELECT 
        c.CLAIM_STATUS,
        COUNT(*) AS TOTAL_CLAIMS,
        SUM(f.TOTAL_PAYMENT_AMOUNT) AS TOTAL_PAYMENT
    FROM {GOLD_FACT_CLAIMS} f
    JOIN {GOLD_DIM_CLAIM} c ON f.CLM_ID = c.CLM_ID
    GROUP BY c.CLAIM_STATUS
    """)
    
    target_table = GOLD_MV_CLAIM_STATUS
    
    mv_claim_status_summary.write.format("delta") \
        .mode(GOLD_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)
    
    mv_count = mv_claim_status_summary.count()
    
    print(f"✓ {target_table}")
    print(f"  Records: {mv_count:,}\n")
except Exception as e:
    print(f"✗ mv_claim_status_summary failed: {str(e)}")
    raise

In [0]:
try:
    print("="*80)
    print("GOLD TABLES")
    print("="*80)
    print()

    gold_tables = [
        GOLD_DIM_PATIENT,
        GOLD_DIM_DISEASE,
        GOLD_DIM_CLAIM,
        GOLD_DIM_GEOGRAPHY,
        GOLD_DIM_DATE,
        GOLD_FACT_CLAIMS
    ]

    for table_name in gold_tables:
        if table_name == GOLD_DIM_PATIENT:
            # SCD Type 2 table - show total and current
            total_count = spark.table(table_name).count()
            current_count = spark.table(table_name).filter(F.col("IS_CURRENT") == True).count()
            print(f"  {table_name}: {current_count:,} current | {total_count:,} total (SCD Type 2)")
        else:
            count = spark.table(table_name).count()
            print(f"  {table_name}: {count:,} records")

    print()
    print("="*80)
    print("MATERIALIZED VIEWS")
    print("="*80)
    print()

    mv_tables = [
        GOLD_MV_STATE,
        GOLD_MV_RISK,
        GOLD_MV_CLAIM_STATUS
    ]

    for mv_name in mv_tables:
        count = spark.table(mv_name).count()
        print(f"  {mv_name}: {count:,} records")

    print()
    print("="*80)
    print("GOLD LAYER COMPLETED SUCCESSFULLY")
    print("="*80)
except Exception as e:
    print(f"✗ Summary failed: {str(e)}")
    raise

In [0]:
try:
    gold_end = datetime.now()
    gold_duration = (gold_end - gold_start).seconds

    fact_count = spark.table(GOLD_FACT_CLAIMS).count()
    dim_patient = spark.table(GOLD_DIM_PATIENT).filter(F.col("IS_CURRENT") == True).count()
    dim_disease = spark.table(GOLD_DIM_DISEASE).count()
    dim_claim = spark.table(GOLD_DIM_CLAIM).count()
    dim_geography = spark.table(GOLD_DIM_GEOGRAPHY).count()
    dim_date = spark.table(GOLD_DIM_DATE).count()

    audit_record = [{
        "run_id": "gold_run",
        "layer": "Gold",
        "partition_num": "all",
        "input_records": gold_input,
        "output_records": fact_count,
        "pass_count": fact_count,
        "fail_count": 0,
        "duration_seconds": gold_duration,
        "status": "SUCCESS",
        "error_message": "",
        "run_timestamp": gold_end
    }]

    audit_df = spark.createDataFrame(audit_record)
    audit_df = audit_df.withColumn("duration_seconds", F.col("duration_seconds").cast(IntegerType()))
    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_LOG_TABLE)

    print(f"{'='*50}")
    print(f"GOLD AUDIT SUMMARY")
    print(f"{'='*50}")
    print(f"Input Records : {gold_input:,}")
    print(f"dim_patient   : {dim_patient:,} (current)")
    print(f"dim_disease   : {dim_disease:,}")
    print(f"dim_claim     : {dim_claim:,}")
    print(f"dim_geography : {dim_geography:,}")
    print(f"dim_date      : {dim_date:,}")
    print(f"fact_claims   : {fact_count:,}")
    print(f"Duration      : {gold_duration} sec")
    print(f"Status        : ✅ SUCCESS")
    print(f"Audit Saved   : pipeline_log ✅")
    print(f"{'='*50}")

except Exception as e:
    audit_record = [{
        "run_id": "gold_run",
        "layer": "Gold",
        "partition_num": "all",
        "input_records": gold_input,
        "output_records": 0,
        "pass_count": 0,
        "fail_count": 0,
        "duration_seconds": 0,
        "status": "FAILED",
        "error_message": str(e),
        "run_timestamp": datetime.now()
    }]
    audit_df = spark.createDataFrame(audit_record)
    audit_df = audit_df.withColumn("duration_seconds", F.col("duration_seconds").cast(IntegerType()))
    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_LOG_TABLE)
    print(f"✗ Gold FAILED: {str(e)}")
    raise